this code is written heavily copied from andrej karphathy from the video and he was open soures code on github and video on yt plz consider it . dataset then reading and explaration data


In [ ]:
#dataset
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


In [ ]:
with open('shakespare.txt', 'r') as f:
    data = f.read()
    

In [ ]:
print("length of dataset in characters: ", len(data))


In [ ]:
print(data[:1000])

In [ ]:
chars = sorted(list(set(data)))
vocab_size = len(chars)
print('Characters:', ''.join(chars))
print('Vocabulary size:', vocab_size)

In [ ]:
stoi  = {ch:i for i , ch in enumerate (chars)}
itos = { i:ch for i, ch in enumerate (chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("abcdefg"))
print(decode(encode("abcdefg")))

In [ ]:
import torch 
lata = torch.tensor(encode(data), dtype=torch.long)
print(lata.shape, lata.dtype)
print(lata[:1000])

In [ ]:
n = int(0.9 * len(lata))
train_data = lata[:n]
val_data = lata[n:]

In [ ]:
block_size = 8 
train_data[:block_size+1]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print (f"when the input is {context} and the target is {target}")

In [ ]:
torch.manual_seed(1337)


batch_size = 4
block_size = 8
def get_batch(split):
    lata = train_data if split == 'train' else val_data
    ix = torch.randint(len(lata)- block_size, (batch_size,))

    x = torch.stack([lata[i:i + block_size] for i in ix])
    y = torch.stack([lata[i + 1:i + 1 + block_size] for i in ix])
    return x, y


xb ,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('\ntargets:')
print(yb.shape) 
print(yb)
print('----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()}the target : {target.item()}")
#wtf is xb and yb? : xb is the input batch and yb is the target batch
#where is the loss function? :



In [ ]:
print(xb)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)
        if targets is None:
            return logits
        B, T, C = logits.shape
        logits = logits.view(B * T, C)  # Flatten to (B*T, C)
        targets = targets.view(B * T)  # Flatten to (B*T,)
        loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits = self(idx)  # Get logits (no targets)
            logits = logits[:, -1, :]  # Last time step
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
# this generated using random predictions, so it will be gibberish. We will train the model to make it better.


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [ ]:
batch_size = 32
for steps in range(1000):
    xb, yb = get_batch('train')
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none= True)
    loss.backward()
    optimizer.step()
xb, yb = get_batch('train')
logits, loss = m(xb, yb)
print(loss.item())


In [ ]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 1000)[0].tolist()))

maths in self attentions

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)

x.shape


In [ ]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow [b,t] = torch.mean(xprev, 0)
        


In [ ]:
x[1]

In [ ]:
x[0]

In [ ]:
xbow[0]